# 13 DBSCAN

依赖安装说明：`pip install numpy matplotlib scikit-learn`

DBSCAN 是基于密度的聚类算法。它不需要预先指定簇数，并且可以识别噪声点。


## 1. 数学逻辑

DBSCAN 有两个关键参数：

- `eps`：邻域半径。
- `min_samples`：成为核心点需要的最少邻居数。

如果一个点的 eps 邻域内至少有 `min_samples` 个点，它就是核心点。簇从核心点开始，把密度可达的点连起来。无法归入任何簇的点标记为噪声 `-1`。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, _ = make_moons(n_samples=260, noise=0.08, random_state=42)
noise = np.random.uniform(low=[-1.8, -1.0], high=[2.8, 1.6], size=(25, 2))
X = np.vstack([X, noise])
X_s = StandardScaler().fit_transform(X)


In [ ]:
# 从零实现：简化版 DBSCAN，重点看核心逻辑

def region_query(X, i, eps):
    d = np.sqrt(((X - X[i]) ** 2).sum(axis=1))
    return np.where(d <= eps)[0]

def dbscan_simple(X, eps=0.25, min_samples=5):
    labels = np.full(len(X), -99)  # -99 表示未访问，-1 表示噪声
    cluster_id = 0
    for i in range(len(X)):
        if labels[i] != -99:
            continue
        neighbors = list(region_query(X, i, eps))
        if len(neighbors) < min_samples:
            labels[i] = -1
            continue
        labels[i] = cluster_id
        queue = neighbors[:]
        while queue:
            j = queue.pop(0)
            if labels[j] == -1:
                labels[j] = cluster_id
            if labels[j] != -99:
                continue
            labels[j] = cluster_id
            j_neighbors = list(region_query(X, j, eps))
            if len(j_neighbors) >= min_samples:
                queue.extend(j_neighbors)
        cluster_id += 1
    return labels

labels_simple = dbscan_simple(X_s, eps=0.25, min_samples=5)
print('从零 DBSCAN 标签:', sorted(set(labels_simple)))


In [ ]:
model = DBSCAN(eps=0.25, min_samples=5)
labels = model.fit_predict(X_s)
print('sklearn DBSCAN 标签:', sorted(set(labels)))
print('噪声点数量:', np.sum(labels == -1))

plt.scatter(X_s[:,0], X_s[:,1], c=labels, cmap='tab10', s=24)
plt.title('DBSCAN 可以发现非球形簇并标记噪声')
plt.show()


## 2. 常见误区

- `eps` 很敏感，太小会产生大量噪声，太大会把簇连在一起。
- 特征尺度会影响距离，通常需要标准化。
- 不同密度的簇会让 DBSCAN 很难同时处理好。

## 3. 小实验

- 改 `eps`，观察簇数量和噪声点数量。
- 改 `min_samples`，观察核心点条件变化。
- 对未标准化数据直接跑，观察差异。
